In [ ]:
import pandas as pd
matids = ["U2240731L"]
"""
matids = [
    "U1234567A",
    "U7654321B",
    "U2345678C",
    "U8765432D",
    "U3456789E",
    "U9876543F",
    "U4567890G",
    "U0987654H",
    "U5678901I",
    "U1098765J",
    "U6789012K",
    "U2109876L",
    "U7890123M",
    "U3210987N",
    "U8901234O",
    "U4321098P",
    "U9012345Q",
    "U5432109R",
    "U1122334S",
    "U9988776T",
    "U2340793K",
    "U2340246H",
    "U2221633B",
    "U2240731L",
]
"""
CSV_PATH = "ResalePricesSingapore.csv"
PRICE_THRESHOLD = 4725

DIGIT_TO_TOWN = {
    '0': 'BEDOK', '1': 'BUKIT PANJANG', '2': 'CLEMENTI',
    '3': 'CHOA CHU KANG', '4': 'HOUGANG', '5': 'JURONG WEST',
    '6': 'PASIR RIS', '7': 'TAMPINES', '8': 'WOODLANDS', '9': 'YISHUN'
}
DIGIT_TO_YEAR = {
    '5': 2015, '6': 2016, '7': 2017, '8': 2018, '9': 2019,
    '0': 2020, '1': 2021, '2': 2022, '3': 2023, '4': 2024
}

def decode_matid(matid):
    digits = [c for c in matid if c.isdigit()]
    target_year = DIGIT_TO_YEAR[digits[-1]]
    start_month = 10 if digits[-2] == '0' else int(digits[-2])
    towns = list(dict.fromkeys(DIGIT_TO_TOWN[d] for d in digits if d in DIGIT_TO_TOWN))
    return target_year, start_month, towns

def run_query(matid, df):
    target_year, start_month, towns = decode_matid(matid)
    results = []
    for x in range(1, 9):
        end_month_num = start_month + x - 1
        if end_month_num <= 12:
            end_year, end_month = target_year, end_month_num
        else:
            end_year, end_month = target_year + 1, end_month_num - 12
        if end_year == target_year:
            mask_time = ((df['YEAR'] == target_year) &
                        (df['MONTH_NUM'] >= start_month) &
                        (df['MONTH_NUM'] <= end_month))
        else:
            mask_time = (((df['YEAR'] == target_year) & (df['MONTH_NUM'] >= start_month)) |
                        ((df['YEAR'] == end_year) & (df['MONTH_NUM'] <= end_month)))
        df_x = df[mask_time & df['TOWN'].isin(towns)].copy()
        for y in range(80, 151): 
            df_xy = df_x[(df_x['FLOOR_AREA'] >= y) & (df_x['RESALE_PRICE'] > 0) & (df_x['FLOOR_AREA'] > 0)].copy()
            if df_xy.empty:
                results.append({
                    '(x, y)': f'({x}, {y})',
                    'Year': 'No result',
                    'Month': '',
                    'Town': '',
                    'Block': '',
                    'Floor_Area': '',
                    'Flat_Model': '',
                    'Lease_Commence_Date': '',
                    'Price_Per_Square_Meter': ''
                })
                continue
            df_xy['PRICE_PER_SQM'] = df_xy['RESALE_PRICE'] / df_xy['FLOOR_AREA']
            min_ppsm = df_xy['PRICE_PER_SQM'].min()
            if min_ppsm > PRICE_THRESHOLD:
                results.append({
                    '(x, y)': f'({x}, {y})',
                    'Year': 'No result',
                    'Month': '',
                    'Town': '',
                    'Block': '',
                    'Floor_Area': '',
                    'Flat_Model': '',
                    'Lease_Commence_Date': '',
                    'Price_Per_Square_Meter': ''
                })
                continue
            row = df_xy.loc[df_xy['PRICE_PER_SQM'].idxmin()]
            results.append({
                '(x, y)': f'({x}, {y})',
                'Year': int(row['YEAR']),
                'Month': f"{int(row['MONTH_NUM']):02d}",
                'Town': row['TOWN'],
                'Block': row['BLOCK'],
                'Floor_Area': int(row['FLOOR_AREA']),
                'Flat_Model': row['FLAT_MODEL'],
                'Lease_Commence_Date': int(row['LEASE_COMMENCE_DATE']),
                'Price_Per_Square_Meter': round(min_ppsm)
            })

    return pd.DataFrame(results)
for MATID in matids:
    df = pd.read_csv(CSV_PATH)
    df.columns = [c.strip().upper() for c in df.columns]
    df = df.rename(columns={'FLOOR_AREA_SQM': 'FLOOR_AREA'})
    df['PARSED_DATE'] = pd.to_datetime(df['MONTH'], format='%b-%y')
    df['YEAR'] = df['PARSED_DATE'].dt.year
    df['MONTH_NUM'] = df['PARSED_DATE'].dt.month

    result_df = run_query(MATID, df)
    output_filename = f"Expected_ScanResult_{MATID}.csv"
    result_df.to_csv(output_filename, index=False)
    print(f"Validation output saved as: {output_filename}")

Validation output saved as: Expected_ScanResult_U1234567A.csv
Validation output saved as: Expected_ScanResult_U7654321B.csv
Validation output saved as: Expected_ScanResult_U2345678C.csv
Validation output saved as: Expected_ScanResult_U8765432D.csv
Validation output saved as: Expected_ScanResult_U3456789E.csv
Validation output saved as: Expected_ScanResult_U9876543F.csv
Validation output saved as: Expected_ScanResult_U4567890G.csv
Validation output saved as: Expected_ScanResult_U0987654H.csv
Validation output saved as: Expected_ScanResult_U5678901I.csv
Validation output saved as: Expected_ScanResult_U1098765J.csv
Validation output saved as: Expected_ScanResult_U6789012K.csv
Validation output saved as: Expected_ScanResult_U2109876L.csv
Validation output saved as: Expected_ScanResult_U7890123M.csv
Validation output saved as: Expected_ScanResult_U3210987N.csv
Validation output saved as: Expected_ScanResult_U8901234O.csv
Validation output saved as: Expected_ScanResult_U4321098P.csv
Validati